# RAG Assignment: Baseline Retrieval vs. Cross-Encoder Re-Ranking

This notebook builds both retrieval pipelines over the same clause-aware chunks from the UNDP *General Terms and Conditions for Contracts*. The current section prepares and validates the shared corpus.

**Controlled comparison:** both pipelines will use the same PDF, chunks, initial Top-10 candidates, LLM, and answer prompt. Only the final Top-3 selection changes.

## 1. Environment setup

In Kaggle, select **Settings → Accelerator → GPU** before running the notebook. The PDF must be attached using **File → Upload input**.

In [1]:
%pip install -q pymupdf sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.9/68.9 MB 22.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 55.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 56.4 MB/s eta 0:00:00:00:0100:01
Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import os
import re
import statistics
from pathlib import Path

# Public Hugging Face models do not require authentication. Keep their routine
# download notices quiet so actual preprocessing warnings remain visible.
os.environ.setdefault("HF_HUB_VERBOSITY", "error")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")

import pymupdf
import torch
from transformers import AutoTokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


## 2. Locate and extract the PDF

The file is discovered recursively instead of relying on a Kaggle dataset slug. This avoids failures when the uploaded input is renamed.

In [3]:
pdf_files = sorted(Path("/kaggle/input").rglob("*.pdf"))

if not pdf_files:
    raise FileNotFoundError(
        "No PDF was found under /kaggle/input. Use File → Upload input first."
    )

undp_matches = [
    path for path in pdf_files
    if "undp" in path.name.lower() and "terms" in path.name.lower()
]

if len(undp_matches) != 1:
    print("PDF files found:")
    for path in pdf_files:
        print(" -", path)
    raise RuntimeError(
        f"Expected one UNDP terms PDF, found {len(undp_matches)}. "
        "Adjust the filename filter in this cell."
    )

PDF_PATH = undp_matches[0]
print("Selected PDF:", PDF_PATH)

Selected PDF: /kaggle/input/datasets/madhamanish/undp-tnc/UNDP-LaoPDR-General Terms and Conditions for Contracts (Goods and-or Services).pdf


In [4]:
with pymupdf.open(PDF_PATH) as document:
    raw_pages = [
        {
            "pdf_page": page_index + 1,
            "text": page.get_text("text"),
        }
        for page_index, page in enumerate(document)
    ]

print("PDF pages:", len(raw_pages))
print("Raw extracted words:", sum(len(p["text"].split()) for p in raw_pages))
print("\nFirst-page sample:\n")
print(raw_pages[0]["text"][:1200])

PDF pages: 25
Raw extracted words: 11083

First-page sample:

 
  
  
  
  
GENERAL TERMS AND CONDITIONS FOR  
CONTRACTS  
This Contract is between the United Nations Development Programme, a subsidiary organ of the United 
Nations established by the General Assembly of the United Nations (hereinafter “UNDP”), on the one 
hand, and a company or organization indicated in the Face Sheet of this Contract (hereinafter the 
“Contractor”), on the other hand.  
1. LEGAL STATUS OF THE PARTIES: UNDP and the Contractor shall be referred to as a “Party” or, 
collectively, “Parties” hereunder, and:  
1.1 Pursuant, inter alia, to the Charter of the United Nations and the Convention on the Privileges and 
Immunities of the United Nations, the United Nations, including its subsidiary organs, has full 
juridical personality and enjoys such privileges and immunities as are necessary for the 
independent fulfillment of its purposes.  
1.2 The Contractor shall have the legal status of an independent cont

## 3. Clean repeated page furniture

Only repeated headers, footers, page labels, soft hyphens, and empty pages are removed. Article and subclause numbering is preserved because it carries legal meaning and supports traceable retrieval results.

In [5]:
FOOTER_PATTERNS = [
    re.compile(r"^REV\.?\s*:\s*SEPTEMBER\s+2017$", re.IGNORECASE),
    re.compile(r"^UNDP\s+GENERAL\s+TERMS\s+AND\s+CONDITIONS\s+FOR\s+CONTRACTS$", re.IGNORECASE),
    re.compile(r"^PAGE(?:\s+\d+)?$", re.IGNORECASE),
]

def clean_page_text(text):
    text = text.replace("\u00ad", "")
    lines = []

    for raw_line in text.splitlines():
        line = re.sub(r"[ \t]+", " ", raw_line).strip()
        if not line:
            lines.append("")
            continue
        if any(pattern.fullmatch(line) for pattern in FOOTER_PATTERNS):
            continue
        lines.append(line)

    cleaned = "\n".join(lines)
    cleaned = re.sub(r"(?<=\w)-\n(?=\w)", "", cleaned)
    cleaned = re.sub(r"\n{3,}", "\n\n", cleaned)
    return cleaned.strip()

clean_pages = []
for page in raw_pages:
    cleaned_text = clean_page_text(page["text"])
    # The final visual page contains no substantive contract text.
    if cleaned_text and re.search(r"[A-Za-z]{3}", cleaned_text):
        clean_pages.append({**page, "text": cleaned_text})

print("Substantive pages retained:", len(clean_pages))
print("Cleaned words:", sum(len(p["text"].split()) for p in clean_pages))
assert len(clean_pages) >= 24, "Too many pages were removed during cleaning."

Substantive pages retained: 24
Cleaned words: 10790


## 4. Detect the 36 major articles

The parser uses the document's numbered, uppercase article headings. Page spans are retained as metadata.

In [6]:
parts = []
page_spans = []
cursor = 0

for page in clean_pages:
    if parts:
        parts.append("\n")
        cursor += 1
    start = cursor
    parts.append(page["text"])
    cursor += len(page["text"])
    page_spans.append((start, cursor, page["pdf_page"]))

full_text = "".join(parts)

ARTICLE_RE = re.compile(
    r"(?m)^[ \t]*(\d{1,2})\.[ \t]+"
    r"([A-Z][A-Z0-9 ,/&()'’;\-\n]{2,}?):"
)

article_matches = list(ARTICLE_RE.finditer(full_text))
articles = []

for index, match in enumerate(article_matches):
    start = match.start()
    end = article_matches[index + 1].start() if index + 1 < len(article_matches) else len(full_text)
    article_text = full_text[start:end].strip()
    source_pages = [
        page_number
        for page_start, page_end, page_number in page_spans
        if max(start, page_start) < min(end, page_end)
    ]
    articles.append({
        "article": int(match.group(1)),
        "heading": re.sub(r"\s+", " ", match.group(2)).strip(),
        "pages": source_pages,
        "text": article_text,
    })

detected_numbers = [article["article"] for article in articles]
expected_numbers = list(range(1, 37))

print("Detected articles:", detected_numbers)
assert detected_numbers == expected_numbers, (
    "Article detection failed. Expected articles 1–36 in order. "
    f"Detected: {detected_numbers}"
)

Detected articles: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36]


## 5. Build clause-aware chunks

A complete article stays together when it fits. Long articles are divided at numbered subclause boundaries. The parent article heading is repeated in every part; token overlap is used only as a fallback when a single subclause itself exceeds the limit.

In [7]:
BI_ENCODER_NAME = "BAAI/bge-small-en-v1.5"
RERANKER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
MAX_CHUNK_TOKENS = 420
FALLBACK_OVERLAP_TOKENS = 40

tokenizer = AutoTokenizer.from_pretrained(BI_ENCODER_NAME)

def token_ids(text):
    # Long source articles are intentionally counted before being split.
    # verbose=False suppresses the premature model-limit warning; final chunks
    # are still enforced below by the MAX_CHUNK_TOKENS assertion.
    return tokenizer.encode(
        text,
        add_special_tokens=False,
        truncation=False,
        verbose=False,
    )

def token_count(text):
    return len(token_ids(text))

def token_windows(text, max_tokens, overlap_tokens):
    ids = token_ids(text)
    if len(ids) <= max_tokens:
        return [text.strip()]
    if overlap_tokens >= max_tokens:
        raise ValueError("Overlap must be smaller than the token window.")

    step = max_tokens - overlap_tokens
    windows = []
    for start in range(0, len(ids), step):
        window_ids = ids[start:start + max_tokens]
        windows.append(tokenizer.decode(window_ids, skip_special_tokens=True).strip())
        if start + max_tokens >= len(ids):
            break
    return windows


In [8]:
def split_article(article):
    article_number = article["article"]
    heading = f'{article_number}. {article["heading"]}:'
    text = article["text"]

    if token_count(text) <= MAX_CHUNK_TOKENS:
        return [text]

    subclause_re = re.compile(
        rf"(?m)^[ \t]*{article_number}\.\d+(?:\.\d+)*[ \t]+"
    )
    matches = list(subclause_re.finditer(text))

    if not matches:
        payload_limit = MAX_CHUNK_TOKENS - token_count(heading) - 4
        return [
            f"{heading}\n{window}"
            for window in token_windows(
                text, payload_limit, FALLBACK_OVERLAP_TOKENS
            )
        ]

    # Keep introductory text with the first numbered subclause, but remove the
    # article heading because it is added consistently to every split part.
    segments = []
    prefix = text[:matches[0].start()].strip()
    prefix = ARTICLE_RE.sub("", prefix, count=1).strip()
    for index, match in enumerate(matches):
        end = matches[index + 1].start() if index + 1 < len(matches) else len(text)
        segment = text[match.start():end].strip()
        if index == 0 and prefix:
            segment = f"{prefix}\n{segment}"
        segments.append(segment)

    chunks = []
    current = heading

    for segment in segments:
        candidate = f"{current}\n{segment}"
        if token_count(candidate) <= MAX_CHUNK_TOKENS:
            current = candidate
            continue

        if current != heading:
            chunks.append(current.strip())
            current = heading

        if token_count(f"{heading}\n{segment}") <= MAX_CHUNK_TOKENS:
            current = f"{heading}\n{segment}"
        else:
            payload_limit = MAX_CHUNK_TOKENS - token_count(heading) - 4
            for window in token_windows(
                segment, payload_limit, FALLBACK_OVERLAP_TOKENS
            ):
                chunks.append(f"{heading}\n{window}".strip())

    if current != heading:
        chunks.append(current.strip())

    return chunks

chunks = []
for article in articles:
    article_parts = split_article(article)
    for part_number, chunk_text in enumerate(article_parts, start=1):
        chunks.append({
            "chunk_id": f'article_{article["article"]:02d}_part_{part_number:02d}',
            "article": article["article"],
            "heading": article["heading"],
            "part": part_number,
            "pages": article["pages"],
            "token_count": token_count(chunk_text),
            "text": chunk_text,
        })


## 6. Validate and inspect the corpus

Do not create embeddings until these checks pass. They catch missing articles, accidental blank chunks, and model-input truncation.

In [9]:
chunk_sizes = [chunk["token_count"] for chunk in chunks]

assert chunks, "No chunks were produced."
assert all(chunk["text"].strip() for chunk in chunks), "Blank chunk detected."
assert max(chunk_sizes) <= MAX_CHUNK_TOKENS, "A chunk exceeds the configured limit."
assert {chunk["article"] for chunk in chunks} == set(range(1, 37))
assert not any(
    "REV.: SEPTEMBER 2017" in chunk["text"].upper()
    for chunk in chunks
), "Repeated footer text remains in the corpus."

print("Total chunks:", len(chunks))
print("Minimum tokens:", min(chunk_sizes))
print("Maximum tokens:", max(chunk_sizes))
print("Mean tokens:", round(statistics.mean(chunk_sizes), 1))
print("Median tokens:", statistics.median(chunk_sizes))

Total chunks: 55
Minimum tokens: 45
Maximum tokens: 419
Mean tokens: 252.6
Median tokens: 264


In [10]:
def show_article_chunks(article_number):
    selected = [c for c in chunks if c["article"] == article_number]
    print(f"Article {article_number}: {len(selected)} chunk(s)")
    for chunk in selected:
        print("\n" + "=" * 80)
        print(chunk["chunk_id"], "| pages", chunk["pages"], "| tokens", chunk["token_count"])
        print("-" * 80)
        print(chunk["text"])

# Article 9 should preserve the assignment rule and its reorganization exception.
show_article_chunks(9)

Article 9: 1 chunk(s)

article_09_part_01 | pages [6] | tokens 302
--------------------------------------------------------------------------------
9. ASSIGNMENT:
9.1
Except as provided in Article 9.2, below, the Contractor may not assign, transfer, pledge or
make any other disposition of the Contract, of any part of the Contract, or of any of the rights,
claims or obligations under the Contract except with the prior written authorization of UNDP.
Any such unauthorized assignment, transfer, pledge or other disposition, or any attempt to do
so, shall not be binding on UNDP. Except as permitted with respect to any approved
subcontractors, the Contractor shall not delegate any of its obligations under this Contract,
except with the prior written consent of UNDP. Any such unauthorized delegation, or attempt
to do so, shall not be binding on UNDP.
9.2
The Contractor may assign or otherwise transfer the Contract to the surviving entity resulting
from a reorganization of the Contractor’s oper

In [11]:
OUTPUT_PATH = Path("/kaggle/working/undp_clause_chunks.json")
OUTPUT_PATH.write_text(
    json.dumps(chunks, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print("Saved validated chunks to:", OUTPUT_PATH)

Saved validated chunks to: /kaggle/working/undp_clause_chunks.json


## 7. Build the baseline dense-vector index

The chunk corpus is embedded once with the Bi-Encoder and added to an exact FAISS inner-product index. Because the embeddings are L2-normalized, inner product is equivalent to cosine similarity. Index construction is reported separately and is not included in per-query retrieval latency.

In [12]:
import time

import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer


def synchronize_device():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


bi_encoder = SentenceTransformer(BI_ENCODER_NAME, device=DEVICE)
print("Bi-Encoder:", BI_ENCODER_NAME)
print("Maximum model sequence length:", bi_encoder.max_seq_length)

Bi-Encoder: BAAI/bge-small-en-v1.5
Maximum model sequence length: 512


In [13]:
document_texts = [chunk["text"] for chunk in chunks]

# Reserve space for the model's special tokens and prevent silent truncation.
assert max(chunk_sizes) + 2 <= bi_encoder.max_seq_length

synchronize_device()
index_start = time.perf_counter()

chunk_embeddings = bi_encoder.encode(
    document_texts,
    batch_size=16,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)
chunk_embeddings = np.asarray(chunk_embeddings, dtype="float32")

vector_index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
vector_index.add(chunk_embeddings)

synchronize_device()
indexing_ms = (time.perf_counter() - index_start) * 1000

embedding_norms = np.linalg.norm(chunk_embeddings, axis=1)
assert chunk_embeddings.shape[0] == len(chunks)
assert vector_index.ntotal == len(chunks)
assert np.allclose(embedding_norms, 1.0, atol=1e-3)

print("Indexed chunks:", vector_index.ntotal)
print("Embedding dimensions:", chunk_embeddings.shape[1])
print(f"One-time indexing latency: {indexing_ms:.2f} ms")

Indexed chunks: 55
Embedding dimensions: 384
One-time indexing latency: 949.21 ms


## 8. Baseline Top-10 retrieval

BGE models use a retrieval instruction on the query only. The stored document chunks remain unchanged. The function returns ranked chunks, cosine-equivalent scores, and total dense-retrieval latency (query encoding plus FAISS search).

In [14]:
QUERY_INSTRUCTION = "Represent this sentence for searching relevant passages: "


def encode_query(query):
    return bi_encoder.encode(
        QUERY_INSTRUCTION + query.strip(),
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).astype("float32")


def retrieve_top_k(query, k=10):
    if not query or not query.strip():
        raise ValueError("Query must not be blank.")
    if not 1 <= k <= len(chunks):
        raise ValueError(f"k must be between 1 and {len(chunks)}.")

    synchronize_device()
    start = time.perf_counter()

    query_embedding = encode_query(query)
    scores, indices = vector_index.search(query_embedding[None, :], k)

    synchronize_device()
    latency_ms = (time.perf_counter() - start) * 1000

    results = []
    for rank, (chunk_index, score) in enumerate(
        zip(indices[0], scores[0]), start=1
    ):
        chunk = chunks[int(chunk_index)]
        results.append({
            "baseline_rank": rank,
            "bi_score": float(score),
            "chunk_index": int(chunk_index),
            **chunk,
        })

    return results, latency_ms


def retrieval_table(results, preview_characters=180):
    rows = []
    for result in results:
        preview = re.sub(r"\s+", " ", result["text"]).strip()
        rows.append({
            "rank": result["baseline_rank"],
            "chunk_id": result["chunk_id"],
            "article": result["article"],
            "heading": result["heading"],
            "pages": result["pages"],
            "cosine_score": round(result["bi_score"], 4),
            "preview": preview[:preview_characters] + ("..." if len(preview) > preview_characters else ""),
        })
    return pd.DataFrame(rows)

In [15]:
# Warm up the model before measuring retrieval latency.
_ = encode_query("warm-up query")

SAMPLE_QUERY = (
    "Under what conditions may the contractor assign the contract "
    "following a corporate reorganization?"
)

baseline_top10, baseline_latency_ms = retrieve_top_k(SAMPLE_QUERY, k=10)
baseline_top3 = baseline_top10[:3]

print("Query:", SAMPLE_QUERY)
print(f"Baseline retrieval latency: {baseline_latency_ms:.2f} ms")
print("Baseline Top-3 chunk IDs:", [r["chunk_id"] for r in baseline_top3])

retrieval_table(baseline_top10)

Query: Under what conditions may the contractor assign the contract following a corporate reorganization?
Baseline retrieval latency: 40.83 ms
Baseline Top-3 chunk IDs: ['article_09_part_01', 'article_03_part_01', 'article_20_part_03']


,rank,chunk_id,article,heading,pages,cosine_score,preview
0,1,article_09_part_01,9,ASSIGNMENT,[6],0.7835,9. ASSIGNMENT: 9.1 Except as provided in Artic...
1,2,article_03_part_01,3,LONG TERM AGREEMENT,[2],0.7165,3. LONG TERM AGREEMENT: If the Contractor is e...
2,3,article_20_part_03,20,TERMINATION,"[17, 18]",0.7085,20. TERMINATION: 20.5.5 the Contractor offers ...
3,4,article_08_part_02,8,RESPONSIBILITY FOR EMPLOYEES,"[4, 5, 6]",0.7075,8. RESPONSIBILITY FOR EMPLOYEES: 8.5 Requireme...
4,5,article_32_part_01,32,OBSERVANCE OF THE LAW,"[21, 22]",0.7060,32. OBSERVANCE OF THE LAW: The Contractor shal...
5,6,article_10_part_01,10,SUBCONTRACTING,[6],0.7057,10. SUBCONTRACTING: In the event that the Cont...
6,7,article_08_part_01,8,RESPONSIBILITY FOR EMPLOYEES,"[4, 5, 6]",0.7016,8. RESPONSIBILITY FOR EMPLOYEES: To the extent...
7,8,article_11_part_01,11,PURCHASE OF GOODS,"[6, 7, 8, 9]",0.7001,11. PURCHASE OF GOODS: to the extent that the ...
8,9,article_02_part_01,2,OBLIGATIONS OF THE CONTRACTOR,"[1, 2]",0.6922,2. OBLIGATIONS OF THE CONTRACTOR: 2.1 The Cont...
9,10,article_08_part_03,8,RESPONSIBILITY FOR EMPLOYEES,"[4, 5, 6]",0.6867,8. RESPONSIBILITY FOR EMPLOYEES: 8.6 Nothing i...


## 9. Cross-Encoder re-ranking

The Cross-Encoder receives the original query and each of the exact ten baseline candidates together. Its scores are used only to re-sort those candidates; it does not search the full corpus. Raw Bi-Encoder and Cross-Encoder scores are on different scales and should not be compared directly.

In [16]:
from sentence_transformers import CrossEncoder

CROSS_ENCODER_MAX_LENGTH = 512

cross_encoder = CrossEncoder(
    RERANKER_NAME,
    device=DEVICE,
    max_length=CROSS_ENCODER_MAX_LENGTH,
)

print("Cross-Encoder:", RERANKER_NAME)
print("Maximum pair length:", CROSS_ENCODER_MAX_LENGTH)

Cross-Encoder: cross-encoder/ms-marco-MiniLM-L-6-v2
Maximum pair length: 512


In [17]:
def rerank_candidates(query, candidates):
    if len(candidates) != 10:
        raise ValueError(
            f"The assignment requires exactly 10 candidates; received {len(candidates)}."
        )

    pairs = [(query, candidate["text"]) for candidate in candidates]
    pair_lengths = [
        len(
            cross_encoder.tokenizer(
                query,
                candidate["text"],
                add_special_tokens=True,
                truncation=False,
                verbose=False,
            )["input_ids"]
        )
        for candidate in candidates
    ]
    assert max(pair_lengths) <= CROSS_ENCODER_MAX_LENGTH, (
        "A query-chunk pair exceeds the Cross-Encoder input limit and would be truncated."
    )

    synchronize_device()
    start = time.perf_counter()

    cross_scores = cross_encoder.predict(
        pairs,
        batch_size=10,
        show_progress_bar=False,
        convert_to_numpy=True,
    )

    synchronize_device()
    latency_ms = (time.perf_counter() - start) * 1000

    cross_scores = np.asarray(cross_scores).reshape(-1)
    reranked_order = np.argsort(-cross_scores)

    reranked = []
    for reranked_rank, candidate_position in enumerate(reranked_order, start=1):
        candidate = candidates[int(candidate_position)]
        baseline_rank = candidate["baseline_rank"]
        reranked.append({
            **candidate,
            "cross_score": float(cross_scores[int(candidate_position)]),
            "reranked_rank": reranked_rank,
            # Positive values indicate promotion; negative values indicate demotion.
            "rank_change": baseline_rank - reranked_rank,
        })

    return reranked, latency_ms


def rank_shift_table(reranked_results, preview_characters=140):
    rows = []
    for result in reranked_results:
        preview = re.sub(r"\s+", " ", result["text"]).strip()
        rows.append({
            "chunk_id": result["chunk_id"],
            "article": result["article"],
            "heading": result["heading"],
            "baseline_rank": result["baseline_rank"],
            "reranked_rank": result["reranked_rank"],
            "rank_change": result["rank_change"],
            "bi_score": round(result["bi_score"], 4),
            "cross_score": round(result["cross_score"], 4),
            "baseline_top3": result["baseline_rank"] <= 3,
            "reranked_top3": result["reranked_rank"] <= 3,
            "preview": preview[:preview_characters] + ("..." if len(preview) > preview_characters else ""),
        })
    return pd.DataFrame(rows)

In [18]:
# Warm up the Cross-Encoder so model initialization is excluded from latency.
_ = cross_encoder.predict(
    [(SAMPLE_QUERY, baseline_top10[0]["text"])],
    show_progress_bar=False,
)

reranked_top10, reranking_latency_ms = rerank_candidates(
    SAMPLE_QUERY, baseline_top10
)
reranked_top3 = reranked_top10[:3]
advanced_total_latency_ms = baseline_latency_ms + reranking_latency_ms

print("Query:", SAMPLE_QUERY)
print("Baseline Top-3:", [r["chunk_id"] for r in baseline_top3])
print("Re-ranked Top-3:", [r["chunk_id"] for r in reranked_top3])
print(f"Dense retrieval latency: {baseline_latency_ms:.2f} ms")
print(f"Cross-Encoder latency: {reranking_latency_ms:.2f} ms")
print(f"Advanced pipeline total: {advanced_total_latency_ms:.2f} ms")

rank_shift_table(reranked_top10)

Query: Under what conditions may the contractor assign the contract following a corporate reorganization?
Baseline Top-3: ['article_09_part_01', 'article_03_part_01', 'article_20_part_03']
Re-ranked Top-3: ['article_09_part_01', 'article_03_part_01', 'article_08_part_02']
Dense retrieval latency: 40.83 ms
Cross-Encoder latency: 79.00 ms
Advanced pipeline total: 119.82 ms


,chunk_id,article,heading,baseline_rank,reranked_rank,rank_change,bi_score,cross_score,baseline_top3,reranked_top3,preview
0,article_09_part_01,9,ASSIGNMENT,1,1,0,0.7835,4.7971,True,True,9. ASSIGNMENT: 9.1 Except as provided in Artic...
1,article_03_part_01,3,LONG TERM AGREEMENT,2,2,0,0.7165,1.0027,True,True,3. LONG TERM AGREEMENT: If the Contractor is e...
2,article_08_part_02,8,RESPONSIBILITY FOR EMPLOYEES,4,3,1,0.7075,0.4771,False,True,8. RESPONSIBILITY FOR EMPLOYEES: 8.5 Requireme...
3,article_11_part_01,11,PURCHASE OF GOODS,8,4,4,0.7001,0.0326,False,False,11. PURCHASE OF GOODS: to the extent that the ...
4,article_20_part_03,20,TERMINATION,3,5,-2,0.7085,-0.5430,True,False,20. TERMINATION: 20.5.5 the Contractor offers ...
5,article_08_part_01,8,RESPONSIBILITY FOR EMPLOYEES,7,6,1,0.7016,-0.6315,False,False,8. RESPONSIBILITY FOR EMPLOYEES: To the extent...
6,article_08_part_03,8,RESPONSIBILITY FOR EMPLOYEES,10,7,3,0.6867,-3.0153,False,False,8. RESPONSIBILITY FOR EMPLOYEES: 8.6 Nothing i...
7,article_02_part_01,2,OBLIGATIONS OF THE CONTRACTOR,9,8,1,0.6922,-4.0136,False,False,2. OBLIGATIONS OF THE CONTRACTOR: 2.1 The Cont...
8,article_32_part_01,32,OBSERVANCE OF THE LAW,5,9,-4,0.7060,-4.3417,False,False,32. OBSERVANCE OF THE LAW: The Contractor shal...
9,article_10_part_01,10,SUBCONTRACTING,6,10,-4,0.7057,-4.8547,False,False,10. SUBCONTRACTING: In the event that the Cont...


## 10. Reusable two-stage retrieval function

This helper will be used unchanged for every stress-test query. It preserves both rankings and the two latency components needed for the final analysis.

In [19]:
def run_two_stage_retrieval(query):
    baseline_results, dense_latency_ms = retrieve_top_k(query, k=10)
    reranked_results, cross_latency_ms = rerank_candidates(
        query, baseline_results
    )

    return {
        "query": query,
        "baseline_top10": baseline_results,
        "baseline_top3": baseline_results[:3],
        "reranked_top10": reranked_results,
        "reranked_top3": reranked_results[:3],
        "dense_latency_ms": dense_latency_ms,
        "reranking_latency_ms": cross_latency_ms,
        "advanced_total_latency_ms": dense_latency_ms + cross_latency_ms,
    }

## 11. Ground-truth stress test

The query bank below targets exceptions, negation, deadlines, and easily confused obligations. Each question is labelled before retrieval with an expected article and a short verbatim answer anchor. The anchor is used only to locate the ground-truth chunk or equivalent overlapping chunks; it is never included in the query.

A qualifying failure case must satisfy both conditions required for a meaningful re-ranking demonstration: the best-ranked ground-truth chunk is at baseline rank 4–10, and an answer-bearing ground-truth chunk moves into the re-ranked Top 3. If none enters the initial Top-10, the query is reported separately as a candidate-generation miss because the Cross-Encoder never receives a relevant candidate.

In [20]:
STRESS_TESTS = [
    {
        "test_id": "payment_not_acceptance",
        "query": "Does UNDP's payment mean that the contractor has fulfilled its duties or that the delivered work has been accepted?",
        "expected_article": 4,
        "answer_anchor": "nor as acceptance by UNDP of the Contractor’s delivery of the Goods and/or provision of the Services",
    },
    {
        "test_id": "advance_security_threshold",
        "query": "When is financial security required before a large upfront amount can be released, and what must that security cover?",
        "expected_article": 5,
        "answer_anchor": "advance payment representing 20% or more",
    },
    {
        "test_id": "replacement_delay_exception",
        "query": "If UNDP asks to replace personnel for reasons unrelated to default, misconduct, or inability to work with UNDP staff, is the contractor liable for the resulting delay?",
        "expected_article": 8,
        "answer_anchor": "not based upon a default by or failure",
    },
    {
        "test_id": "criminal_charge_notice",
        "query": "How quickly must UNDP be told when a worker with access to its premises is charged with a serious offence?",
        "expected_article": 8,
        "answer_anchor": "Within one working day after learning",
    },
    {
        "test_id": "reorganization_transfer",
        "query": "May the agreement pass to a successor after a merger even without ordinary prior authorization, and which safeguards must be met?",
        "expected_article": 9,
        "answer_anchor": "surviving entity resulting from a reorganization",
    },
    {
        "test_id": "subcontractor_responsibility",
        "query": "After UNDP approves another firm to perform part of the work, does responsibility for that firm's performance shift away from the main contractor?",
        "expected_article": 10,
        "answer_anchor": "solely responsible for all services and obligations performed by its subcontractors",
    },
    {
        "test_id": "delivery_not_acceptance",
        "query": "Does physical handover of purchased items by itself establish that UNDP has accepted them?",
        "expected_article": 11,
        "answer_anchor": "Delivery of the Goods shall not be deemed in itself as constituting acceptance",
    },
    {
        "test_id": "minimum_warranty_period",
        "query": "What is the minimum time the product warranty continues after UNDP accepts the items?",
        "expected_article": 11,
        "answer_anchor": "not less than one (1) year following acceptance",
    },
    {
        "test_id": "ip_indemnity_exception",
        "query": "When is the contractor's intellectual-property infringement indemnity unavailable because UNDP dictated a specification change?",
        "expected_article": 12,
        "answer_anchor": "compliance with specific written instructions by UNDP",
    },
    {
        "test_id": "insurance_not_liability_cap",
        "query": "Does carrying the required insurance or a stated deductible place a ceiling on the contractor's contractual liability?",
        "expected_article": 13,
        "answer_anchor": "shall in any way be construed as limiting",
    },
    {
        "test_id": "equipment_wear_exception",
        "query": "Must equipment supplied by UNDP be returned in its original condition, and which deterioration does not require compensation?",
        "expected_article": 15,
        "answer_anchor": "same condition as when delivered to the Contractor",
    },
    {
        "test_id": "preexisting_ip",
        "query": "Does UNDP own technology the contractor developed independently before the work, or does it receive a different right?",
        "expected_article": 16,
        "answer_anchor": "pre-existed the performance by the Contractor",
    },
    {
        "test_id": "publicity_permission",
        "query": "Can the supplier advertise its relationship with UNDP or use the UN name and emblem for commercial goodwill?",
        "expected_article": 17,
        "answer_anchor": "commercial advantage or goodwill",
    },
    {
        "test_id": "confidentiality_prior_knowledge",
        "query": "Do confidentiality restrictions prevent disclosure of information already known independently or received from an unrestricted third party?",
        "expected_article": 18,
        "answer_anchor": "obtained by the Recipient from a third party without restriction",
    },
    {
        "test_id": "force_majeure_long_stop",
        "query": "After how long may UNDP treat a force-majeure interruption as permanent inability to perform?",
        "expected_article": 19,
        "answer_anchor": "period in excess of ninety (90) days",
    },
    {
        "test_id": "nonwaiver_by_inaction",
        "query": "If a party does not exercise an available remedy, has it surrendered that right or released the parties from their duties?",
        "expected_article": 21,
        "answer_anchor": "shall not be deemed for any purposes to constitute a waiver",
    },
    {
        "test_id": "no_minimum_purchase",
        "query": "Is UNDP committed to buy a minimum volume, or prevented from purchasing equivalent services elsewhere?",
        "expected_article": 22,
        "answer_anchor": "no obligation to purchase any minimum quantities",
    },
    {
        "test_id": "punitive_damages",
        "query": "May arbitrators impose punitive damages when deciding an unresolved contractual dispute?",
        "expected_article": 23,
        "answer_anchor": "shall have no authority to award punitive damages",
    },
    {
        "test_id": "tax_paid_without_approval",
        "query": "Will UNDP reimburse a tax that the contractor paid without first consulting UNDP and obtaining written authorization to pay under protest?",
        "expected_article": 25,
        "answer_anchor": "specifically authorized the Contractor to pay such taxes",
    },
    {
        "test_id": "audit_survival_period",
        "query": "For how many years after expiration or early termination can paid invoices still be examined?",
        "expected_article": 27,
        "answer_anchor": "Each invoice paid by UNDP shall be subject to a post-payment audit",
    },
    {
        "test_id": "limitations_indemnity_exception",
        "query": "Which category of obligation is carved out from the three-year deadline for beginning arbitration?",
        "expected_article": 28,
        "answer_anchor": "Except with respect to any indemnification obligations",
    },
    {
        "test_id": "minor_marriage_exception",
        "query": "Is the under-eighteen rule applied when the contractor's worker is legally married to the younger person under the worker's country-of-citizenship law?",
        "expected_article": 35,
        "answer_anchor": "shall not apply the foregoing standard relating to age",
    },
]

def normalize_for_match(text):
    # PDF extractors can vary in spacing and punctuation, e.g. (90) vs 90
    # or curly vs straight apostrophes. Match the same word sequence instead.
    return re.sub(r"[^a-z0-9]+", " ", text.lower()).strip()

def locate_ground_truth_chunks(test):
    anchor = normalize_for_match(test["answer_anchor"])
    matches = [
        chunk for chunk in chunks
        if chunk["article"] == test["expected_article"]
        and anchor in normalize_for_match(chunk["text"])
    ]
    assert matches, (
        f'{test["test_id"]}: no ground-truth chunk contains the answer anchor'
    )
    return matches

for test in STRESS_TESTS:
    ground_truth_chunks = locate_ground_truth_chunks(test)
    test["expected_chunk_ids"] = [
        chunk["chunk_id"] for chunk in ground_truth_chunks
    ]

print(f"Validated {len(STRESS_TESTS)} known-answer stress tests.")
pd.DataFrame(STRESS_TESTS)[
    ["test_id", "expected_article", "expected_chunk_ids", "query"]
]

from IPython.display import display

evaluation_runs = {}
evaluation_rows = []

for test in STRESS_TESTS:
    run = run_two_stage_retrieval(test["query"])
    evaluation_runs[test["test_id"]] = run
    expected_ids = set(test["expected_chunk_ids"])

    baseline_hit = next(
        (item for item in run["baseline_top10"] if item["chunk_id"] in expected_ids),
        None,
    )
    reranked_hit = next(
        (item for item in run["reranked_top10"] if item["chunk_id"] in expected_ids),
        None,
    )

    baseline_rank = baseline_hit["baseline_rank"] if baseline_hit else None
    reranked_rank = reranked_hit["reranked_rank"] if reranked_hit else None
    baseline_failure = baseline_rank is not None and 4 <= baseline_rank <= 10
    qualifying_failure = (
        baseline_failure and reranked_rank is not None and reranked_rank <= 3
    )

    if baseline_rank is None:
        outcome = "candidate miss"
    elif baseline_rank <= 3:
        outcome = "baseline success"
    elif qualifying_failure:
        outcome = "re-ranking fixed"
    else:
        outcome = "still outside Top-3"

    evaluation_rows.append({
        "test_id": test["test_id"],
        "expected_chunk_ids": ", ".join(test["expected_chunk_ids"]),
        "baseline_rank": baseline_rank,
        "reranked_rank": reranked_rank,
        "rank_gain": (baseline_rank - reranked_rank)
        if baseline_rank is not None and reranked_rank is not None else None,
        "outcome": outcome,
        "qualifying_failure": qualifying_failure,
        "dense_ms": run["dense_latency_ms"],
        "rerank_ms": run["reranking_latency_ms"],
        "advanced_total_ms": run["advanced_total_latency_ms"],
    })

evaluation_table = pd.DataFrame(evaluation_rows)
for column in ["baseline_rank", "reranked_rank", "rank_gain"]:
    evaluation_table[column] = evaluation_table[column].astype("Int64")

evaluation_table = evaluation_table.sort_values(
    ["qualifying_failure", "baseline_rank"],
    ascending=[False, False],
    na_position="last",
).reset_index(drop=True)

print("Outcome counts:")
print(evaluation_table["outcome"].value_counts().to_string())
print("\nMean measured latency across the stress-test bank:")
print(f'  Baseline retrieval: {evaluation_table["dense_ms"].mean():.2f} ms')
print(f'  Cross-Encoder only: {evaluation_table["rerank_ms"].mean():.2f} ms')
print(f'  Advanced total:     {evaluation_table["advanced_total_ms"].mean():.2f} ms')

display(evaluation_table[[
    "test_id", "expected_chunk_ids", "baseline_rank",
    "reranked_rank", "rank_gain", "outcome",
]])

Validated 22 known-answer stress tests.
Outcome counts:
outcome
baseline success       18
re-ranking fixed        3
still outside Top-3     1

Mean measured latency across the stress-test bank:
  Baseline retrieval: 10.94 ms
  Cross-Encoder only: 48.87 ms
  Advanced total:     59.81 ms


,test_id,expected_chunk_ids,baseline_rank,reranked_rank,rank_gain,outcome
0,delivery_not_acceptance,article_11_part_01,5,2,3,re-ranking fixed
1,subcontractor_responsibility,article_10_part_01,4,2,2,re-ranking fixed
2,minimum_warranty_period,article_11_part_04,4,3,1,re-ranking fixed
3,payment_not_acceptance,article_04_part_01,5,4,1,still outside Top-3
4,ip_indemnity_exception,article_12_part_01,2,2,0,baseline success
5,confidentiality_prior_knowledge,article_18_part_02,2,1,1,baseline success
6,punitive_damages,article_23_part_01,2,1,1,baseline success
7,advance_security_threshold,article_05_part_01,1,8,-7,baseline success
8,replacement_delay_exception,article_08_part_02,1,1,0,baseline success
9,criminal_charge_notice,article_08_part_03,1,1,0,baseline success


In [21]:
qualified_case_ids = evaluation_table.loc[
    evaluation_table["qualifying_failure"], "test_id"
].tolist()

print(f"Qualifying re-ranking failure cases found: {len(qualified_case_ids)}")
print(qualified_case_ids)

if len(qualified_case_ids) < 2:
    print(
        "\nFewer than two qualifying cases were found. Do not relabel baseline successes "
        "or candidate misses as re-ranking fixes. Add or revise grounded queries, then rerun."
    )
else:
    selected_failure_case_ids = qualified_case_ids[:2]
    print("\nSelected for the final Answer 1 vs Answer 2 comparison:", selected_failure_case_ids)

    for test_id in selected_failure_case_ids:
        test = next(item for item in STRESS_TESTS if item["test_id"] == test_id)
        run = evaluation_runs[test_id]
        print("\n" + "=" * 100)
        print("Test:", test_id)
        print("Question:", test["query"])
        print("Ground-truth chunks:", test["expected_chunk_ids"])
        display(rank_shift_table(run["reranked_top10"]))

Qualifying re-ranking failure cases found: 3
['delivery_not_acceptance', 'subcontractor_responsibility', 'minimum_warranty_period']

Selected for the final Answer 1 vs Answer 2 comparison: ['delivery_not_acceptance', 'subcontractor_responsibility']

Test: delivery_not_acceptance
Question: Does physical handover of purchased items by itself establish that UNDP has accepted them?
Ground-truth chunks: ['article_11_part_01']


,chunk_id,article,heading,baseline_rank,reranked_rank,rank_change,bi_score,cross_score,baseline_top3,reranked_top3,preview
0,article_11_part_05,11,PURCHASE OF GOODS,1,1,0,0.7410,1.2537,True,True,11. PURCHASE OF GOODS: 11.5.8 The Contractor s...
1,article_11_part_01,11,PURCHASE OF GOODS,5,2,3,0.6931,0.0677,False,True,11. PURCHASE OF GOODS: to the extent that the ...
2,article_11_part_04,11,PURCHASE OF GOODS,4,3,1,0.6989,-1.5934,False,True,11. PURCHASE OF GOODS: 11.5.2 If the Contracto...
3,article_22_part_01,22,NON-EXCLUSIVITY,3,4,-1,0.7071,-2.0668,True,False,22. NON-EXCLUSIVITY: Unless otherwise specifie...
4,article_11_part_06,11,PURCHASE OF GOODS,2,5,-3,0.7090,-2.3424,True,False,11. PURCHASE OF GOODS: 11.7.4 pay all costs re...
5,article_03_part_01,3,LONG TERM AGREEMENT,9,6,3,0.6715,-3.0002,False,False,3. LONG TERM AGREEMENT: If the Contractor is e...
6,article_36_part_01,36,ANTI-TERRORISM,10,7,3,0.6640,-3.8643,False,False,36. ANTI-TERRORISM: The Contractor agrees to u...
7,article_04_part_03,4,PRICE AND PAYMENT,7,8,-1,0.6764,-4.0016,False,False,4. PRICE AND PAYMENT: 4.2.5 Payments effected ...
8,article_12_part_03,12,INDEMNIFICATION,8,9,-1,0.6755,-4.4024,False,False,12. INDEMNIFICATION: 12.5.2 replace or modify ...
9,article_15_part_01,15,EQUIPMENT FURNISHED BY UNDP TO THE CONTRACTOR,6,10,-4,0.6859,-4.8299,False,False,15. EQUIPMENT FURNISHED BY UNDP TO THE CONTRAC...



Test: subcontractor_responsibility
Question: After UNDP approves another firm to perform part of the work, does responsibility for that firm's performance shift away from the main contractor?
Ground-truth chunks: ['article_10_part_01']


,chunk_id,article,heading,baseline_rank,reranked_rank,rank_change,bi_score,cross_score,baseline_top3,reranked_top3,preview
0,article_08_part_02,8,RESPONSIBILITY FOR EMPLOYEES,3,1,2,0.7545,2.2033,True,True,8. RESPONSIBILITY FOR EMPLOYEES: 8.5 Requireme...
1,article_10_part_01,10,SUBCONTRACTING,4,2,2,0.7521,1.9481,False,True,10. SUBCONTRACTING: In the event that the Cont...
2,article_08_part_03,8,RESPONSIBILITY FOR EMPLOYEES,1,3,-2,0.7676,1.8113,True,True,8. RESPONSIBILITY FOR EMPLOYEES: 8.6 Nothing i...
3,article_30_part_01,30,SOURCE OF INSTRUCTIONS,10,4,6,0.7302,0.3851,False,False,30. SOURCE OF INSTRUCTIONS: The Contractor sha...
4,article_08_part_01,8,RESPONSIBILITY FOR EMPLOYEES,2,5,-3,0.7663,-0.1056,True,False,8. RESPONSIBILITY FOR EMPLOYEES: To the extent...
5,article_08_part_04,8,RESPONSIBILITY FOR EMPLOYEES,5,6,-1,0.7464,-0.5782,False,False,8. RESPONSIBILITY FOR EMPLOYEES: 8.9 All opera...
6,article_31_part_01,31,STANDARDS OF CONDUCT,7,7,0,0.7380,-1.1095,False,False,31. STANDARDS OF CONDUCT: The Contractor warra...
7,article_27_part_01,27,AUDITS AND INVESTIGATIONS,9,8,1,0.7304,-1.2210,False,False,27. AUDITS AND INVESTIGATIONS: 27.1 Each invoi...
8,article_04_part_03,4,PRICE AND PAYMENT,8,9,-1,0.7354,-2.0501,False,False,4. PRICE AND PAYMENT: 4.2.5 Payments effected ...
9,article_32_part_01,32,OBSERVANCE OF THE LAW,6,10,-4,0.7381,-3.0598,False,False,32. OBSERVANCE OF THE LAW: The Contractor shal...


## 12. Controlled Answer 1 vs. Answer 2 generation

For each selected failure case, **Answer 1** uses the baseline Top-3 and **Answer 2** uses the re-ranked Top-3. Both calls use the same deterministic instruction model, system instruction, question template, context formatting, and generation settings. Therefore, the retrieved context is the only experimental variable.

The 1.5B-parameter generator fits comfortably on a Kaggle T4 alongside the two retrieval models and supports the complete three-chunk context without truncation.

In [22]:
from transformers import AutoModelForCausalLM, AutoTokenizer

GENERATOR_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
GENERATION_MAX_NEW_TOKENS = 180

generator_tokenizer = AutoTokenizer.from_pretrained(GENERATOR_NAME)
generator_dtype = torch.float16 if DEVICE == "cuda" else torch.float32
generator_model = AutoModelForCausalLM.from_pretrained(
    GENERATOR_NAME,
    dtype=generator_dtype,
).to(DEVICE)
generator_model.eval()

if generator_tokenizer.pad_token_id is None:
    generator_tokenizer.pad_token_id = generator_tokenizer.eos_token_id

print("Generator:", GENERATOR_NAME)
print("Generator device:", next(generator_model.parameters()).device)
print("Generator dtype:", next(generator_model.parameters()).dtype)

Generator: Qwen/Qwen2.5-1.5B-Instruct
Generator device: cuda:0
Generator dtype: torch.float16


In [23]:
ANSWER_SYSTEM_PROMPT = (
    "You are a careful contract question-answering assistant. "
    "Answer using only the supplied context. If the context does not contain "
    "the answer, state exactly: The supplied context does not provide the answer. "
    "Do not use outside knowledge. Preserve important conditions and exceptions. "
    "Return only the concise answer text. Do not invent or mention chunk identifiers; "
    "the notebook records verified evidence separately."
)

def format_answer_context(retrieved_top3):
    blocks = []
    for result in retrieved_top3:
        label = (
            f'[{result["chunk_id"]} | Article {result["article"]}: '
            f'{result["heading"]}]'
        )
        blocks.append(f'{label}\n{result["text"]}')
    return "\n\n".join(blocks)

def build_answer_prompt(query, retrieved_top3):
    context = format_answer_context(retrieved_top3)
    messages = [
        {"role": "system", "content": ANSWER_SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (
                f"Question:\n{query}\n\n"
                f"Context:\n{context}\n\n"
                "Give one concise answer using only this context."
            ),
        },
    ]
    return generator_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

def generate_grounded_answer(query, retrieved_top3):
    prompt = build_answer_prompt(query, retrieved_top3)
    model_inputs = generator_tokenizer(
        prompt, return_tensors="pt", add_special_tokens=False
    ).to(DEVICE)
    input_tokens = int(model_inputs["input_ids"].shape[1])

    model_limit = getattr(generator_model.config, "max_position_embeddings", None)
    if model_limit is not None:
        assert input_tokens + GENERATION_MAX_NEW_TOKENS <= model_limit, (
            "The complete prompt would exceed the generator context window."
        )

    synchronize_device()
    start = time.perf_counter()
    with torch.inference_mode():
        generated = generator_model.generate(
            **model_inputs,
            max_new_tokens=GENERATION_MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=generator_tokenizer.pad_token_id,
            eos_token_id=generator_tokenizer.eos_token_id,
        )
    synchronize_device()
    latency_ms = (time.perf_counter() - start) * 1000

    new_token_ids = generated[0, input_tokens:]
    answer = generator_tokenizer.decode(
        new_token_ids, skip_special_tokens=True
    ).strip()
    return {
        "answer": answer,
        "input_tokens": input_tokens,
        "output_tokens": int(new_token_ids.shape[0]),
        "generation_latency_ms": latency_ms,
    }

In [24]:
# Warm up the generator so one-time CUDA initialization is excluded from comparison.
warmup_prompt = generator_tokenizer(
    "Answer briefly: ready?", return_tensors="pt"
).to(DEVICE)
with torch.inference_mode():
    _ = generator_model.generate(
        **warmup_prompt,
        max_new_tokens=4,
        do_sample=False,
        pad_token_id=generator_tokenizer.pad_token_id,
    )
synchronize_device()

assert len(selected_failure_case_ids) >= 2
answer_comparisons = []

for test_id in selected_failure_case_ids:
    test = next(item for item in STRESS_TESTS if item["test_id"] == test_id)
    run = evaluation_runs[test_id]
    expected_ids = set(test["expected_chunk_ids"])

    answer_1 = generate_grounded_answer(
        test["query"], run["baseline_top3"]
    )
    answer_2 = generate_grounded_answer(
        test["query"], run["reranked_top3"]
    )

    baseline_ids = [item["chunk_id"] for item in run["baseline_top3"]]
    reranked_ids = [item["chunk_id"] for item in run["reranked_top3"]]
    baseline_evidence_ids = sorted(expected_ids.intersection(baseline_ids))
    reranked_evidence_ids = sorted(expected_ids.intersection(reranked_ids))

    answer_comparisons.append({
        "test_id": test_id,
        "query": test["query"],
        "ground_truth_chunk_ids": sorted(expected_ids),
        "baseline_top3_ids": baseline_ids,
        "reranked_top3_ids": reranked_ids,
        "baseline_verified_evidence_ids": baseline_evidence_ids,
        "reranked_verified_evidence_ids": reranked_evidence_ids,
        "baseline_contains_ground_truth": bool(baseline_evidence_ids),
        "reranked_contains_ground_truth": bool(reranked_evidence_ids),
        "answer_1_baseline": answer_1["answer"],
        "answer_2_reranked": answer_2["answer"],
        "baseline_generation_ms": answer_1["generation_latency_ms"],
        "reranked_generation_ms": answer_2["generation_latency_ms"],
        "baseline_end_to_end_ms": (
            run["dense_latency_ms"] + answer_1["generation_latency_ms"]
        ),
        "advanced_end_to_end_ms": (
            run["advanced_total_latency_ms"]
            + answer_2["generation_latency_ms"]
        ),
        "answer_1_input_tokens": answer_1["input_tokens"],
        "answer_2_input_tokens": answer_2["input_tokens"],
        "answer_1_output_tokens": answer_1["output_tokens"],
        "answer_2_output_tokens": answer_2["output_tokens"],
    })

for comparison in answer_comparisons:
    print("\n" + "=" * 100)
    print("TEST:", comparison["test_id"])
    print("QUESTION:", comparison["query"])
    print("BASELINE TOP-3:", comparison["baseline_top3_ids"])
    print("ANSWER 1 (baseline):", comparison["answer_1_baseline"])
    print("VERIFIED BASELINE EVIDENCE:", comparison["baseline_verified_evidence_ids"])
    print("RE-RANKED TOP-3:", comparison["reranked_top3_ids"])
    print("ANSWER 2 (re-ranked):", comparison["answer_2_reranked"])
    print("VERIFIED RE-RANKED EVIDENCE:", comparison["reranked_verified_evidence_ids"])
    print(
        "Generation latency (ms):",
        f'Answer 1={comparison["baseline_generation_ms"]:.2f},',
        f'Answer 2={comparison["reranked_generation_ms"]:.2f}',
    )


TEST: delivery_not_acceptance
QUESTION: Does physical handover of purchased items by itself establish that UNDP has accepted them?
BASELINE TOP-3: ['article_11_part_05', 'article_11_part_06', 'article_22_part_01']
ANSWER 1 (baseline): The context states that UNDP is not obligated to accept Goods unless and until it has had a reasonable opportunity to inspect them after delivery. Payment alone does not constitute acceptance of the Goods.
VERIFIED BASELINE EVIDENCE: []
RE-RANKED TOP-3: ['article_11_part_05', 'article_11_part_01', 'article_11_part_04']
ANSWER 2 (re-ranked): The physical handover of purchased items by itself does not establish that UNDP has accepted them. Acceptance must occur through written confirmation provided by UNDP after a reasonable opportunity for inspection.
VERIFIED RE-RANKED EVIDENCE: ['article_11_part_01']
Generation latency (ms): Answer 1=1659.65, Answer 2=1698.82

TEST: subcontractor_responsibility
QUESTION: After UNDP approves another firm to perform part 

In [25]:
comparison_path = Path("/kaggle/working/rag_answer_comparison.json")
comparison_path.write_text(
    json.dumps(answer_comparisons, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

comparison_summary = pd.DataFrame(answer_comparisons)[[
    "test_id",
    "ground_truth_chunk_ids",
    "baseline_top3_ids",
    "reranked_top3_ids",
    "baseline_verified_evidence_ids",
    "reranked_verified_evidence_ids",
    "baseline_contains_ground_truth",
    "reranked_contains_ground_truth",
    "baseline_end_to_end_ms",
    "advanced_end_to_end_ms",
]]

print("Saved answer comparison to:", comparison_path)
display(comparison_summary)

Saved answer comparison to: /kaggle/working/rag_answer_comparison.json


,test_id,ground_truth_chunk_ids,baseline_top3_ids,reranked_top3_ids,baseline_verified_evidence_ids,reranked_verified_evidence_ids,baseline_contains_ground_truth,reranked_contains_ground_truth,baseline_end_to_end_ms,advanced_end_to_end_ms
0,delivery_not_acceptance,[article_11_part_01],"[article_11_part_05, article_11_part_06, artic...","[article_11_part_05, article_11_part_01, artic...",[],[article_11_part_01],False,True,1670.074882,1755.323748
1,subcontractor_responsibility,[article_10_part_01],"[article_08_part_03, article_08_part_01, artic...","[article_08_part_02, article_10_part_01, artic...",[],[article_10_part_01],False,True,1437.584996,1461.747984


## 13. Final quantitative summary

Retrieval metrics are computed over the complete, pre-labelled 22-query stress-test bank. Answer-context evidence rates are reported separately over the two selected failure cases. Keeping these denominators separate avoids overstating the result.

In [26]:
def reciprocal_rank_mean(rank_series):
    ranks = [int(rank) for rank in rank_series.dropna()]
    return float(np.mean([1.0 / rank for rank in ranks])) if ranks else 0.0

baseline_top3_hits = (
    evaluation_table["baseline_rank"].notna()
    & (evaluation_table["baseline_rank"] <= 3)
)
reranked_top3_hits = (
    evaluation_table["reranked_rank"].notna()
    & (evaluation_table["reranked_rank"] <= 3)
)

retrieval_metrics = {
    "number_of_stress_queries": int(len(evaluation_table)),
    "baseline_top3_hits": int(baseline_top3_hits.sum()),
    "reranked_top3_hits": int(reranked_top3_hits.sum()),
    "baseline_top3_hit_rate": float(baseline_top3_hits.mean()),
    "reranked_top3_hit_rate": float(reranked_top3_hits.mean()),
    "baseline_top10_recall": float(evaluation_table["baseline_rank"].notna().mean()),
    "baseline_mrr_at_10": reciprocal_rank_mean(evaluation_table["baseline_rank"]),
    "reranked_mrr_at_10": reciprocal_rank_mean(evaluation_table["reranked_rank"]),
    "qualifying_reranking_fixes": int(evaluation_table["qualifying_failure"].sum()),
}

latency_metrics_ms = {
    "mean_dense_retrieval": float(evaluation_table["dense_ms"].mean()),
    "mean_cross_encoder": float(evaluation_table["rerank_ms"].mean()),
    "mean_advanced_retrieval_total": float(evaluation_table["advanced_total_ms"].mean()),
    "mean_answer_1_generation": float(np.mean([
        item["baseline_generation_ms"] for item in answer_comparisons
    ])),
    "mean_answer_2_generation": float(np.mean([
        item["reranked_generation_ms"] for item in answer_comparisons
    ])),
}

answer_evidence_metrics = {
    "number_of_selected_failure_cases": int(len(answer_comparisons)),
    "baseline_context_evidence_rate": float(np.mean([
        item["baseline_contains_ground_truth"] for item in answer_comparisons
    ])),
    "reranked_context_evidence_rate": float(np.mean([
        item["reranked_contains_ground_truth"] for item in answer_comparisons
    ])),
}

print("Retrieval metrics:")
display(pd.DataFrame([retrieval_metrics]).T.rename(columns={0: "value"}))
print("Latency metrics (milliseconds):")
display(pd.DataFrame([latency_metrics_ms]).T.rename(columns={0: "value"}))
print("Selected-case evidence metrics:")
display(pd.DataFrame([answer_evidence_metrics]).T.rename(columns={0: "value"}))

final_results = {
    "models": {
        "bi_encoder": BI_ENCODER_NAME,
        "cross_encoder": RERANKER_NAME,
        "generator": GENERATOR_NAME,
    },
    "retrieval_metrics": retrieval_metrics,
    "latency_metrics_ms": latency_metrics_ms,
    "answer_evidence_metrics": answer_evidence_metrics,
    "answer_comparisons": answer_comparisons,
}
final_results_path = Path("/kaggle/working/rag_experiment_results.json")
final_results_path.write_text(
    json.dumps(final_results, indent=2, ensure_ascii=False), encoding="utf-8"
)
print("Saved complete experiment results to:", final_results_path)

Retrieval metrics:


,value
number_of_stress_queries,22.000000
baseline_top3_hits,18.000000
reranked_top3_hits,20.000000
baseline_top3_hit_rate,0.818182
reranked_top3_hit_rate,0.909091
baseline_top10_recall,1.000000
baseline_mrr_at_10,0.790909
reranked_mrr_at_10,0.804924
qualifying_reranking_fixes,3.000000


Latency metrics (milliseconds):


,value
mean_dense_retrieval,10.938676
mean_cross_encoder,48.870996
mean_advanced_retrieval_total,59.809673
mean_answer_1_generation,1543.235479
mean_answer_2_generation,1551.434867


Selected-case evidence metrics:


,value
number_of_selected_failure_cases,2.0
baseline_context_evidence_rate,0.0
reranked_context_evidence_rate,1.0


Saved complete experiment results to: /kaggle/working/rag_experiment_results.json


### Latency and scaling interpretation

- The baseline uses an exact FAISS inner-product search. Its storage grows with the number and dimension of chunk vectors, and exact search work grows with corpus size. For a much larger corpus, an approximate FAISS index can reduce search time at the cost of some candidate recall.
- The Cross-Encoder performs a full query-chunk inference for each of the fixed Top-10 candidates. Its cost is therefore roughly proportional to candidate count and sequence length after retrieval, rather than total corpus size. Increasing the candidate pool may improve recall but increases re-ranking latency.
- Batching the ten pairs improves GPU throughput, although it requires more accelerator memory. Re-ranking can also be skipped for simple or latency-critical queries.
- Answer generation is expected to dominate end-to-end latency. Generation time can vary with answer length, so retrieval-only latency is the cleaner measurement of the cost introduced by re-ranking.
- The experiment demonstrates the intended benefit when the initial Top-10 has adequate recall: the Cross-Encoder can promote an answer-bearing passage into the limited Top-3 context supplied to the generator. It cannot recover a relevant passage that is absent from the initial candidate set.

## Next stage

Re-run Sections 12 and 13 so the answers use the citation-free prompt and the final metrics are populated. Then download the executed notebook for the final written conclusions and submission check.

In [29]:
from pathlib import Path
from IPython.display import Markdown, display
import numpy as np

baseline_rank = evaluation_table["baseline_rank"]
reranked_rank = evaluation_table["reranked_rank"]

fixed_rows = evaluation_table[
    evaluation_table["qualifying_failure"]
]

regression_rows = evaluation_table[
    baseline_rank.le(3).fillna(False)
    & ~reranked_rank.le(3).fillna(False)
]

fixed_names = ", ".join(fixed_rows["test_id"]) or "None"
regression_names = ", ".join(regression_rows["test_id"]) or "None"

rank_lines = []
for test_id in selected_failure_case_ids:
    row = evaluation_table[
        evaluation_table["test_id"] == test_id
    ].iloc[0]

    rank_lines.append(
        f"- `{test_id}`: relevant chunk moved from "
        f"rank {int(row['baseline_rank'])} to "
        f"rank {int(row['reranked_rank'])}."
    )

baseline_e2e = float(np.mean([
    item["baseline_end_to_end_ms"]
    for item in answer_comparisons
]))

advanced_e2e = float(np.mean([
    item["advanced_end_to_end_ms"]
    for item in answer_comparisons
]))

e2e_difference = advanced_e2e - baseline_e2e
e2e_percent = (
    100 * e2e_difference / baseline_e2e
    if baseline_e2e else 0.0
)

retrieval_multiplier = (
    latency_metrics_ms["mean_advanced_retrieval_total"]
    / latency_metrics_ms["mean_dense_retrieval"]
)

lines = [
    "## 14. Results, discussion, and conclusion",
    "",
    "### Retrieval results",
    "",
    (
        f"The evaluation used "
        f"**{retrieval_metrics['number_of_stress_queries']}** "
        "pre-labelled contract questions."
    ),
    "",
    (
        f"- Baseline Top-3: "
        f"**{retrieval_metrics['baseline_top3_hits']}/22 "
        f"({retrieval_metrics['baseline_top3_hit_rate']:.1%})**"
    ),
    (
        f"- Re-ranked Top-3: "
        f"**{retrieval_metrics['reranked_top3_hits']}/22 "
        f"({retrieval_metrics['reranked_top3_hit_rate']:.1%})**"
    ),
    (
        f"- Baseline Recall@10: "
        f"**{retrieval_metrics['baseline_top10_recall']:.1%}**"
    ),
    (
        f"- Baseline MRR@10: "
        f"**{retrieval_metrics['baseline_mrr_at_10']:.3f}**"
    ),
    (
        f"- Re-ranked MRR@10: "
        f"**{retrieval_metrics['reranked_mrr_at_10']:.3f}**"
    ),
    (
        f"- Qualifying re-ranking fixes: "
        f"**{retrieval_metrics['qualifying_reranking_fixes']}**"
    ),
    f"- Top-3 regressions: **{len(regression_rows)}**",
    "",
    f"Fixed queries: `{fixed_names}`.",
    "",
    f"Regressed queries: `{regression_names}`.",
    "",
    "### Selected failure cases",
    "",
]

lines.extend(rank_lines)

lines.extend([
    "",
    (
        "For both selected cases, the baseline Top-3 omitted the "
        "answer-bearing chunk, while the re-ranked Top-3 included it."
    ),
    "",
    (
        f"- Baseline verified-evidence rate: "
        f"**{answer_evidence_metrics['baseline_context_evidence_rate']:.0%}**"
    ),
    (
        f"- Re-ranked verified-evidence rate: "
        f"**{answer_evidence_metrics['reranked_context_evidence_rate']:.0%}**"
    ),
    "",
    (
        "The baseline answers were plausible, but the labelled supporting "
        "clauses were absent. Answer 2 was supported by the relevant clauses: "
        "`article_11_part_01` for delivery versus acceptance and "
        "`article_10_part_01` for subcontractor responsibility."
    ),
    "",
    "### Latency trade-off",
    "",
    (
        f"- Mean dense retrieval: "
        f"**{latency_metrics_ms['mean_dense_retrieval']:.2f} ms**"
    ),
    (
        f"- Mean Cross-Encoder cost: "
        f"**{latency_metrics_ms['mean_cross_encoder']:.2f} ms**"
    ),
    (
        f"- Mean advanced retrieval: "
        f"**{latency_metrics_ms['mean_advanced_retrieval_total']:.2f} ms**"
    ),
    f"- Retrieval-stage slowdown: **{retrieval_multiplier:.2f}x**",
    f"- Mean baseline end-to-end latency: **{baseline_e2e:.2f} ms**",
    f"- Mean advanced end-to-end latency: **{advanced_e2e:.2f} ms**",
    (
        f"- End-to-end difference: "
        f"**{e2e_difference:.2f} ms ({e2e_percent:.1f}%)**"
    ),
    "",
    (
        "Generation took roughly 1.5 seconds and dominated total latency. "
        "The re-ranking cost was large relative to retrieval alone but small "
        "relative to the complete answer-generation pipeline."
    ),
    "",
    "### Scaling and limitations",
    "",
    (
        "- Exact FAISS search becomes more expensive as the corpus grows; "
        "an approximate index could improve speed with a possible recall loss."
    ),
    (
        "- Cross-Encoder cost grows with candidate count and sequence length, "
        "not directly with the complete corpus size."
    ),
    (
        "- The Cross-Encoder cannot recover a passage missing from the "
        "initial Top-10 candidate set."
    ),
    "- The experiment uses one document, 55 chunks, and 22 test questions.",
    "- Only two failure cases were used for answer comparison.",
    "",
    "### Conclusion",
    "",
    (
        f"Cross-Encoder re-ranking improved the Top-3 hit rate from "
        f"**{retrieval_metrics['baseline_top3_hit_rate']:.1%}** to "
        f"**{retrieval_metrics['reranked_top3_hit_rate']:.1%}** and supplied "
        "the correct supporting passage for both selected answer comparisons."
    ),
    "",
    (
        "The advanced pipeline provides better grounding at the cost of "
        "additional latency. It is preferable when evidence quality is more "
        "important than minimum retrieval time, while selective re-ranking "
        "may be better for latency-sensitive applications."
    ),
])

report = "\n".join(lines)

display(Markdown(report))

final_report_path = Path(
    "/kaggle/working/final_conclusion.md"
)
final_report_path.write_text(report, encoding="utf-8")

print("Saved final conclusion to:", final_report_path)

## 14. Results, discussion, and conclusion

### Retrieval results

The evaluation used **22** pre-labelled contract questions.

- Baseline Top-3: **18/22 (81.8%)**
- Re-ranked Top-3: **20/22 (90.9%)**
- Baseline Recall@10: **100.0%**
- Baseline MRR@10: **0.791**
- Re-ranked MRR@10: **0.805**
- Qualifying re-ranking fixes: **3**
- Top-3 regressions: **1**

Fixed queries: `delivery_not_acceptance, subcontractor_responsibility, minimum_warranty_period`.

Regressed queries: `advance_security_threshold`.

### Selected failure cases

- `delivery_not_acceptance`: relevant chunk moved from rank 5 to rank 2.
- `subcontractor_responsibility`: relevant chunk moved from rank 4 to rank 2.

For both selected cases, the baseline Top-3 omitted the answer-bearing chunk, while the re-ranked Top-3 included it.

- Baseline verified-evidence rate: **0%**
- Re-ranked verified-evidence rate: **100%**

The baseline answers were plausible, but the labelled supporting clauses were absent. Answer 2 was supported by the relevant clauses: `article_11_part_01` for delivery versus acceptance and `article_10_part_01` for subcontractor responsibility.

### Latency trade-off

- Mean dense retrieval: **10.94 ms**
- Mean Cross-Encoder cost: **48.87 ms**
- Mean advanced retrieval: **59.81 ms**
- Retrieval-stage slowdown: **5.47x**
- Mean baseline end-to-end latency: **1553.83 ms**
- Mean advanced end-to-end latency: **1608.54 ms**
- End-to-end difference: **54.71 ms (3.5%)**

Generation took roughly 1.5 seconds and dominated total latency. The re-ranking cost was large relative to retrieval alone but small relative to the complete answer-generation pipeline.

### Scaling and limitations

- Exact FAISS search becomes more expensive as the corpus grows; an approximate index could improve speed with a possible recall loss.
- Cross-Encoder cost grows with candidate count and sequence length, not directly with the complete corpus size.
- The Cross-Encoder cannot recover a passage missing from the initial Top-10 candidate set.
- The experiment uses one document, 55 chunks, and 22 test questions.
- Only two failure cases were used for answer comparison.

### Conclusion

Cross-Encoder re-ranking improved the Top-3 hit rate from **81.8%** to **90.9%** and supplied the correct supporting passage for both selected answer comparisons.

The advanced pipeline provides better grounding at the cost of additional latency. It is preferable when evidence quality is more important than minimum retrieval time, while selective re-ranking may be better for latency-sensitive applications.

Saved final conclusion to: /kaggle/working/final_conclusion.md
